# Terminal primer removal with Sassy

This prototype removes primers from the adapter- and barcode-cleaned reads in `manual_test/processed`. It uses Sassy's IUPAC-aware approximate matcher with `alpha=0.5`, giving a semi-global, overhang-tolerant search for primers that are partially sequenced at either read boundary.

It supports both read orientations. At the 5' end it searches supplied `*_LEFT` primers for forward reads and supplied `*_RIGHT` primers for reverse reads. At the 3' end it searches reverse complements of `*_RIGHT` primers for forward reads and reverse complements of `*_LEFT` primers for reverse reads. A primer must align within the configured terminal slack; internal matches are ignored.

In [122]:
from __future__ import annotations

import math
import re
from collections import Counter
from collections.abc import Iterator
from dataclasses import dataclass
from itertools import islice
from pathlib import Path

import sassy
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqIO.QualityIO import FastqGeneralIterator

In [124]:
ROOT = Path.cwd()
PRIMER_FASTA = ROOT / "manual_test/ESIB_EQA_2026_SARS1.primers.fasta"
INPUT_FASTQ = ROOT / "manual_test/processed/EQA_04.subsampled.fastq"

TERMINAL_WINDOW = 160
TERMINAL_SLACK = 50
MAX_ERROR_RATE = 0.20
MIN_ALIGNED_PRIMER_FRACTION = 0.55
MIN_ALIGNED_PRIMER_BASES = 12
OVERHANG_PENALTY = 0.5
SEARCHER = sassy.Searcher("iupac", rc=False, alpha=OVERHANG_PENALTY)

## Primer and FASTQ helpers

In [126]:
@dataclass(frozen=True)
class Primer:
    name: str
    sequence: str


@dataclass(frozen=True)
class FastqRecord:
    title: str
    sequence: str
    qualities: str


@dataclass(frozen=True)
class TerminalMatch:
    primer: Primer
    start: int
    end: int
    cost: int
    cigar: str


@dataclass(frozen=True)
class TrimDecision:
    record: FastqRecord
    left_match: TerminalMatch | None
    right_match: TerminalMatch | None


def read_primers(path: Path) -> tuple[tuple[Primer, ...], tuple[Primer, ...]]:
    left_primers = []
    right_primers = []

    # Read each primer record from the FASTA file.
    for record in SeqIO.parse(path, "fasta"):
        sequence = str(record.seq).upper()

        # Store left primers in their original orientation.
        if "_LEFT" in record.id:
            left_primers.append(Primer(record.id, sequence))

        # Reverse-complement right primers for read-oriented matching.
        elif "_RIGHT" in record.id:
            right_primers.append(
                Primer(record.id, str(record.seq.reverse_complement()).upper())
            )
        else:
            raise ValueError(f"Primer name has no LEFT/RIGHT orientation: {record.id}")

    # Ensure that both primer orientations were found.
    if not left_primers or not right_primers:
        raise ValueError("Expected at least one left and one right primer")

    return tuple(left_primers), tuple(right_primers)


def prepare_primer_groups(
    primers: tuple[Primer, ...],
) -> tuple[tuple[tuple[Primer, ...], tuple[bytes, ...]], ...]:
    groups: dict[int, list[Primer]] = {}

    # Group primers by length for efficient Sassy searches.
    for primer in primers:
        groups.setdefault(len(primer.sequence), []).append(primer)

    # Store both Primer objects and their byte-encoded sequences.
    return tuple(
        (
            tuple(group),
            tuple(primer.sequence.encode("ascii") for primer in group),
        )
        for group in groups.values()
    )


def reverse_complement_primers(primers: tuple[Primer, ...]) -> tuple[Primer, ...]:
    # Create primers representing the opposite read orientation.
    return tuple(
        Primer(
            primer.name,
            str(Seq(primer.sequence).reverse_complement()),
        )
        for primer in primers
    )


def read_fastq(path: Path) -> Iterator[FastqRecord]:
    # Read FASTQ records lazily to avoid loading the entire file into memory.
    with path.open() as handle:
        for title, sequence, qualities in FastqGeneralIterator(handle):
            yield FastqRecord(
                title=title,
                sequence=sequence.upper(),
                qualities=qualities,
            )


# Load the original primer orientations.
LEFT_PRIMERS, RIGHT_PRIMERS = read_primers(PRIMER_FASTA)

# Generate primers for reverse-oriented reads.
REVERSE_LEFT_PRIMERS = reverse_complement_primers(LEFT_PRIMERS)
REVERSE_RIGHT_PRIMERS = reverse_complement_primers(RIGHT_PRIMERS)

# Search left and right read boundaries using both orientations.
LEFT_PRIMER_GROUPS = prepare_primer_groups(
    LEFT_PRIMERS + REVERSE_RIGHT_PRIMERS
)
RIGHT_PRIMER_GROUPS = prepare_primer_groups(
    RIGHT_PRIMERS + REVERSE_LEFT_PRIMERS
)

print(
    f"Loaded {len(LEFT_PRIMERS)} left and "
    f"{len(RIGHT_PRIMERS)} right primers across both read orientations."
)
print(LEFT_PRIMER_GROUPS)
print(RIGHT_PRIMER_GROUPS)

Loaded 106 left and 108 right primers across both read orientations.
(((Primer(name='ncov-2019_1_LEFT', sequence='AACAAACCAACCAACTTTCGATCTC'), Primer(name='ncov-2019_9_LEFT', sequence='TCTTCTTAGAGGGAGAAACACTTCC'), Primer(name='ncov-2019_30_LEFT', sequence='ACCTAGAGTTTTTAGTGCAGTTGGT'), Primer(name='ncov-2019_38_LEFT', sequence='GACTGTGTTATGTATGCATCAGCTG'), Primer(name='ncov-2019_43_LEFT', sequence='GGATTTGAAATGGGCTAGATTCCCT'), Primer(name='ncov-2019_54_LEFT', sequence='ACATGATGAGTTAACAGGACACATG'), Primer(name='ncov-2019_64_LEFT', sequence='GCCTATTTTGGAATTGCAATGTCGA'), Primer(name='ncov-2019_70_LEFT', sequence='TTGATTGGTGATTGTGCAACTGTAC'), Primer(name='ncov-2019_84_LEFT', sequence='GTTGATTTAGGTGACATCTCTGGCA'), Primer(name='ncov-2019_91_LEFT', sequence='TCCAGTAGCAGTGACAATATTGCTT'), Primer(name='ncov-2019_94_LEFT', sequence='ACCCGTGTCCTATTCACTTCTATTC'), Primer(name='ncov-2019_98_LEFT', sequence='CCAGGAACTAATCAGACAAGGAACT'), Primer(name='ncov-2019_4_RIGHT', sequence='ACAACAGCATTTTGGGGTAAGTA

## Semi-global terminal matching and trimming

In [128]:
def minimum_aligned_bases(primer: Primer) -> int:
    # Require both the configured minimum and the fraction-based minimum.
    return max(
        MIN_ALIGNED_PRIMER_BASES,
        math.ceil(len(primer.sequence) * MIN_ALIGNED_PRIMER_FRACTION),
    )


def cigar_error_count(cigar: str) -> int:
    # Count substitutions, insertions, and deletions in the CIGAR string.
    return sum(
        int(length)
        for length, operation in re.findall(r"(\d+)([XID])", cigar)
        if operation in "XID"
    )


def maximum_match_cost(primer: Primer, aligned_bases: int) -> int:
    # Allow errors according to the number of aligned bases.
    observed_error_budget = math.ceil(aligned_bases * MAX_ERROR_RATE)

    # Penalize primer bases that extend beyond the aligned read region.
    overhang_bases = max(0, len(primer.sequence) - aligned_bases)

    # Combine mismatch and overhang costs.
    return observed_error_budget + math.ceil(overhang_bases * OVERHANG_PENALTY)


def search_budget(primers: tuple[Primer, ...]) -> int:
    # Use the first primer to determine the maximum search cost.
    primer = primers[0]
    return maximum_match_cost(primer, minimum_aligned_bases(primer))


def match_rank(match: TerminalMatch) -> tuple[float, int, int]:
    # Prefer matches covering a larger fraction of the primer.
    observed_bases = match.end - match.start
    return (
        -(observed_bases / len(match.primer.sequence)),
        cigar_error_count(match.cigar),
        match.cost,
    )


def best_terminal_match(
    window: str,
    primer_groups: tuple[tuple[tuple[Primer, ...], tuple[bytes, ...]], ...],
    terminal: str,
) -> TerminalMatch | None:
    candidates = []
    encoded_window = window.encode("ascii")

    # Search each primer group against the terminal window.
    for primers, patterns in primer_groups:
        for match in SEARCHER.search_many(
            patterns,
            [encoded_window],
            k=search_budget(primers),
            threads=1,
            mode="batch_patterns",
        ):
            primer = primers[match.pattern_idx]
            aligned_bases = match.text_end - match.text_start

            # Keep only matches satisfying alignment and error constraints.
            if (
                aligned_bases >= minimum_aligned_bases(primer)
                and cigar_error_count(match.cigar)
                <= math.ceil(aligned_bases * MAX_ERROR_RATE)
                and match.cost <= maximum_match_cost(primer, aligned_bases)
            ):
                candidates.append((primer, match))

    # Keep matches close to the requested terminal end.
    if terminal == "left":
        candidates = [
            candidate
            for candidate in candidates
            if candidate[1].text_start <= TERMINAL_SLACK
        ]
        placement = lambda candidate: candidate[1].text_start
    elif terminal == "right":
        candidates = [
            candidate
            for candidate in candidates
            if len(window) - candidate[1].text_end <= TERMINAL_SLACK
        ]
        placement = lambda candidate: len(window) - candidate[1].text_end
    else:
        raise ValueError(f"Unknown terminal: {terminal}")

    if not candidates:
        return None

    # Prefer the lowest-cost match, then the one closest to the terminal.
    primer, match = min(
        candidates,
        key=lambda candidate: (candidate[1].cost, placement(candidate)),
    )

    # Convert the searcher's match into the application's match type.
    return TerminalMatch(
        primer=primer,
        start=match.text_start,
        end=match.text_end,
        cost=match.cost,
        cigar=match.cigar,
    )


def trim_record(record: FastqRecord) -> TrimDecision:
    # Restrict searches to the sequence ends.
    left_window = record.sequence[:TERMINAL_WINDOW]
    right_window_start = max(0, len(record.sequence) - TERMINAL_WINDOW)
    right_window = record.sequence[right_window_start:]

    # Find primer matches at both terminals.
    left_match = best_terminal_match(left_window, LEFT_PRIMER_GROUPS, "left")
    right_match = best_terminal_match(right_window, RIGHT_PRIMER_GROUPS, "right")

    # Convert terminal matches into absolute sequence coordinates.
    left_cut = left_match.end if left_match else 0
    right_cut = (
        right_window_start + right_match.start
        if right_match
        else len(record.sequence)
    )

    # Resolve overlapping matches by retaining the better-ranked match.
    if left_cut > right_cut:
        if match_rank(left_match) <= match_rank(right_match):
            right_match = None
            right_cut = len(record.sequence)
        else:
            left_match = None
            left_cut = 0

    # Trim sequence and qualities using the same coordinates.
    return TrimDecision(
        record=FastqRecord(
            title=record.title,
            sequence=record.sequence[left_cut:right_cut],
            qualities=record.qualities[left_cut:right_cut],
        ),
        left_match=left_match,
        right_match=right_match,
    )

## Coordinate checks

These synthetic reads verify the coordinate convention used for trimming before real reads are processed.

In [127]:
left_primer = LEFT_PRIMERS[0]
right_primer = RIGHT_PRIMERS[0]
payload = "GATTACA" * 20
synthetic = FastqRecord(
    title="synthetic",
    sequence=left_primer.sequence + payload + right_primer.sequence,
    qualities="I" * (len(left_primer.sequence) + len(payload) + len(right_primer.sequence)),
)
decision = trim_record(synthetic)
assert decision.left_match is not None
assert decision.right_match is not None
assert decision.left_match.primer == left_primer
assert decision.right_match.primer == right_primer
assert decision.record.sequence == payload
assert len(decision.record.sequence) == len(decision.record.qualities)
print("fake read check passed")

fake read check passed


## Inspect a small sample

This evaluates the first 250 reads from each input without writing output files.

In [129]:
SAMPLE_SIZE = 250

# Read only the first SAMPLE_SIZE records for a quick inspection.
records = list(islice(read_fastq(INPUT_FASTQ), SAMPLE_SIZE))

# Trim both terminal primers from each sampled record.
decisions = [trim_record(record) for record in records]

# Count the primer names detected at the 5' end.
left_calls = Counter(
    decision.left_match.primer.name
    for decision in decisions
    if decision.left_match
)

# Count the primer names detected at the 3' end.
right_calls = Counter(
    decision.right_match.primer.name
    for decision in decisions
    if decision.right_match
)

# Calculate the total number of bases removed from the sample.
bases_removed = sum(
    len(record.sequence) - len(decision.record.sequence)
    for record, decision in zip(records, decisions)
)

# Print summary statistics and the most frequently detected primers.
print(
    f"{INPUT_FASTQ.name}: reads={len(decisions)}, "
    f"left={sum(decision.left_match is not None for decision in decisions)}, "
    f"right={sum(decision.right_match is not None for decision in decisions)}, "
    f"top left={left_calls.most_common(3)}, top right={right_calls.most_common(3)}, "
    f"bases removed={bases_removed}"
)

EQA_04.subsampled.fastq: reads=250, left=245, right=223, top left=[('ncov-2019_1_LEFT', 9), ('ncov-2019_7_LEFT', 8), ('ncov-2019_6_RIGHT', 5)], top right=[('ncov-2019_7_RIGHT', 7), ('ncov-2019_91_RIGHT', 7), ('ncov-2019_6_LEFT', 6)], bases removed=12296


## Write primer-cleaned FASTQs

Running this cell processes the full fastq file. Source files are preserved; output names end in `.noprimers.fastq`.

In [130]:
def output_path_for(input_fastq: Path) -> Path:
    # Replace the input filename suffix with ".noprimers.fastq".
    return input_fastq.with_name(f"{input_fastq.stem}.noprimers.fastq")


def trim_fastq(input_fastq: Path) -> tuple[Path, int, int, int]:
    # Determine where the trimmed FASTQ will be written.
    output_fastq = output_path_for(input_fastq)

    # Initialize processing and trimming counters.
    total_reads = 0
    left_removed = 0
    right_removed = 0
    bases_removed = 0

    # Open the output file and process input records one at a time.
    with output_fastq.open("w") as output_handle:
        for source in read_fastq(input_fastq):
            # Detect and remove terminal primers from the current read.
            decision = trim_record(source)
            cleaned = decision.record

            # Confirm that sequence and quality lengths remain synchronized.
            assert len(cleaned.sequence) == len(cleaned.qualities)

            # Confirm that trimming did not increase the read length.
            assert len(cleaned.sequence) <= len(source.sequence)

            # Write the cleaned record in FASTQ format.
            output_handle.write(
                f"@{cleaned.title}\n{cleaned.sequence}\n+\n{cleaned.qualities}\n"
            )

            # Update read and primer-removal statistics.
            total_reads += 1
            left_removed += decision.left_match is not None
            right_removed += decision.right_match is not None
            bases_removed += len(source.sequence) - len(cleaned.sequence)

    # Report processing statistics after all reads have been written.
    print(
        f"{input_fastq.name} -> {output_fastq.name}: {total_reads:,} reads, "
        f"left={left_removed:,}, right={right_removed:,}, bases removed={bases_removed:,}"
    )

    # Return output location and summary statistics for later validation.
    return output_fastq, total_reads, left_removed + right_removed, bases_removed


# Process the input FASTQ and retain the resulting summary.
results = trim_fastq(INPUT_FASTQ)

EQA_04.subsampled.fastq -> EQA_04.subsampled.noprimers.fastq: 10,000 reads, left=9,586, right=9,166, bases removed=484,916


## Output validation

This verifies that every input record remains in order and that trimming never desynchronizes a sequence from its quality string.

In [118]:
output_fastq, expected_reads, _, expected_removed_bases = results
source_records = read_fastq(INPUT_FASTQ)
cleaned_records = read_fastq(output_fastq)
observed_reads = 0
observed_removed_bases = 0

while True:
    source = next(source_records, None)
    cleaned = next(cleaned_records, None)
    assert (source is None) == (cleaned is None)
    if source is None:
        break
    assert source.title == cleaned.title
    assert len(cleaned.sequence) == len(cleaned.qualities)
    assert len(cleaned.sequence) <= len(source.sequence)
    observed_reads += 1
    observed_removed_bases += len(source.sequence) - len(cleaned.sequence)

assert observed_reads == expected_reads
assert observed_removed_bases == expected_removed_bases
print(f"Validated {output_fastq.name}: {observed_reads:,} records.")

Validated EQA_04.subsampled.noprimers.fastq: 10,000 records.
